# Neuronale Netzwerke - Grundlagen

Wir wollen folgendes Neuronales Netzwerk erstellen:

<img src="images/03-nn-iris.png" width="50%">

Quelle: ChatGPT generiert
- - -

## Erklärung

**Schritt 1 – Die Eingaben**

    x1 = sepal length
    x2 = sepal width
    x3 = petal length
    x4 = petal width

**Schritt 2 – Gewichte**

Jede Eingabe bekommt ein Gewicht.

Das Gewicht sagt im Prinzip:

> Wie wichtig ist dieses Feature (Attribute) für die Entscheidung?

Beispiel:

    w1 = Gewicht für sepal length
    w2 = Gewicht für sepal width
    w3 = Gewicht für petal length
    w4 = Gewicht für petal width

Diese Gewichte werden beim Training gelernt.

**Schritt 3 – Bias**

Der Bias ist einfach ein zusätzlicher Wert, der das Ergebnis verschiebt.
Man kann ihn sich vorstellen wie eine **Konstante** in einer linearen Gleichung.
Beispiel aus der Schule:

    y = 2x + 1

Hier ist **1 der Bias**.

**Schritt 4 – Wie viele Neuronen gibt es?**

Dein Modell hat **3 Neuronen** (in der Mitte)

* Neuron 1 → erkennt eher Setosa
* Neuron 2 → erkennt eher Versicolor
* Neuron 3 → erkennt eher Virginica

**Schritt 5 – Parameter zählen**

Ein Neuron braucht:

    4 Gewichte
    + 1 Bias
    -----------
    = 5 Parameter

Da du **3 Neuronen** hast:

    3 × 5 = 15 Parameter

Dies gilt für die erste Dense-Schicht. Das später verwendete Keras-Modell hat zusätzlich eine Softmax-Ausgabeschicht und damit insgesamt **27 trainierbare Parameter**.


In [ ]:
%%bash
# Optional: nur ausführen, falls Pakete in der aktuellen Jupyter-Umgebung fehlen.
source ~/.jupyter/bin/activate
pip install -U matplotlib numpy scikit-learn pandas tensorflow packaging

In [ ]:
import warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, tensorflow as tf
from packaging.version import Version
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings('ignore')
%matplotlib inline
colors = 'bwr'; CMAP = colors
for package, minimum in [(sklearn, '1.0'), (tf, '2.16'), (pd, '1.5')]:
    print(f'{package.__name__}: {package.__version__}')
    assert Version(package.__version__) >= Version(minimum)
print(f'Keras: {keras.__version__}')

## Iris mit Neuronalen Netzwerken

In [ ]:
iris = load_iris()
print(iris.DESCR)

### Warum macht man diese Visualisierung?

Diese Art Plot dient zur Exploratory Data Analysis (EDA).

Typische Ziele:
* Feature-Verteilungen verstehen
* Korrelationen erkennen
* Klassentrennung visuell prüfen
* Ausreisser identifizieren

Bevor ein neuronales Netz trainiert wird, ist das extrem hilfreich, weil man sieht:
* welche Features informativ sind
* welche redundant sein könnten
* ob das Problem einfach oder schwierig ist

In [ ]:
iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
pd.plotting.scatter_matrix(iris_df, c=iris.target, cmap=CMAP, edgecolors='black', figsize=(20, 20))
# plt.savefig('ML_0701.png', bbox_inches='tight')
plt.show()

## Das künstliche Neuron

Der Abschnitt **„Das künstliche Neuron“** ist konzeptionell wichtig, weil hier der Übergang von klassischem Machine Learning zu neuronalen Netzen erklärt wird. Die Idee ist, dass ein neuronales Netz letztlich aus vielen **einfachen Recheneinheiten (Neuronen)** besteht, die miteinander verbunden sind.

Ein künstliches Neuron bildet im Wesentlichen eine **gewichtete Summe der Eingaben** und wendet danach eine **Aktivierungsfunktion** an.

Formal kann man das so beschreiben:

z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b

Dabei gilt:

* (x_1, x_2, …, x_n) sind die Eingabewerte (Features)
* (w_1, w_2, …, w_n) sind die Gewichte
* (b) ist der Bias-Term
* (z) ist die gewichtete Summe

Diese gewichtete Summe wird danach durch eine Aktivierungsfunktion geschickt:

[
y = f(z)
]

Das Ergebnis (y) ist der **Output des Neurons**.

### Bedeutung der einzelnen Komponenten

Die **Eingaben (Features)** entsprechen den Messwerten der Daten.
Im Iris-Beispiel sind das beispielsweise:

* sepal length
* sepal width
* petal length
* petal width

Die **Gewichte** bestimmen, wie stark ein Feature das Ergebnis beeinflusst.

Beispiel:

* grosses Gewicht → Feature ist wichtig
* kleines Gewicht → Feature hat wenig Einfluss

Der **Bias** verschiebt die Entscheidungsgrenze. Man kann ihn sich als eine Art **Offset** vorstellen.

### Aktivierungsfunktion

Die Aktivierungsfunktion entscheidet, **wie das Neuron auf die gewichtete Summe reagiert**.

Ohne Aktivierungsfunktion wäre das Modell nur eine lineare Funktion und könnte keine komplexeren Muster lernen.

Typische Aktivierungsfunktionen sind:

* ReLU
* Sigmoid
* Tanh
* Softmax

Eine der wichtigsten ist **ReLU (Rectified Linear Unit)**:

f(x)=\max(0,x)

Diese Funktion gibt:

* 0 zurück, wenn der Wert negativ ist
* den Wert selbst, wenn er positiv ist

ReLU ist heute Standard in vielen neuronalen Netzen, weil sie effizient trainiert werden kann.

### Intuition

Man kann sich ein künstliches Neuron wie einen **Entscheidungsfilter** vorstellen.

Es prüft:

* Welche Eingaben sind vorhanden?
* Wie stark sind diese gewichtet?
* Soll ein Signal weitergegeben werden?

Wenn viele solcher Neuronen kombiniert werden, entsteht ein **neuronales Netz**, das komplexe Muster erkennen kann.

Viele solcher Neuronen bilden zusammen ein neuronales Netz. Frameworks wie Keras abstrahieren diese mathematischen Operationen in leicht verwendbare Layer, sodass man komplexe Modelle mit wenigen Zeilen Code definieren kann.




In [ ]:
w0, w1, w2 = 3, -4, 2

def neuron_no_activation(x1, x2):
    return w0 + x1 * w1 + x2 * w2

print('Erste Iris:', iris.data[0])
print('Ohne Aktivierung:', neuron_no_activation(5.1, 3.5))

### Activation Functions

In [ ]:
def centerAxis(uses_negative=False):
    ax = plt.gca()
    ax.spines['left'].set_position('center')
    if uses_negative: ax.spines['bottom'].set_position('center')
    ax.spines['right'].set_color('none')
    ax.spines['top'].set_color('none')
    ax.xaxis.set_ticks_position('bottom')
    ax.yaxis.set_ticks_position('left')

def np_step(x): return 0.5 * (np.sign(x) + 1)
def np_sigmoid(x): return 1 / (1 + np.exp(-x))
def np_relu(x): return np.maximum(0, x)

def plot_activation(function, name, uses_negative=False):
    x = np.arange(-10, 10, 0.01)
    plt.figure(figsize=(8, 4))
    centerAxis(uses_negative)
    plt.plot(x, function(x), lw=3)
    plt.title(name)
    plt.show()

#### Step Function: abrupter, nicht stetig differenzierbarer Übergang zwischen 0 und 1

In [ ]:
plot_activation(np_step, 'Step')

#### Sigmoid Function: Fliessender Übergang zwischen 0 und 1

In [ ]:
plot_activation(np_sigmoid, 'Sigmoid')

#### Tangens Hyperbolicus Function: Fliessender Übergang zwischen -1 und 1

In [ ]:
plot_activation(np.tanh, 'Tangens hyperbolicus')

#### Relu: Einfach zu berechnen, setzt kompletten negativen Wertebereich auf 0

In [ ]:
plot_activation(np_relu, 'ReLU')

In [ ]:
def sigmoid(x): return 1 / (1 + math.exp(-x))
def neuron(x1, x2): return sigmoid(neuron_no_activation(x1, x2))

weights_with_bias = np.array([w0, w1, w2])
def np_neuron(X):
    return np_sigmoid(np.sum(np.concatenate(([1], X)) * weights_with_bias))

print('Sigmoid-Neuron:', neuron(5.1, 3.5))
print('NumPy-Neuron:', np_neuron(np.array([5.1, 3.5])))

## Unser erstes neuronales Netz mit Keras

Der Abschnitt „Unser erstes neuronales Netz mit Keras“ markiert den Übergang von der theoretischen Beschreibung eines künstlichen Neurons zur praktischen Implementierung eines neuronalen Netzes.

Ein neuronales Netz entsteht, indem viele solcher Neuronen in Schichten organisiert und miteinander verbunden werden. Die Ausgabe einer Schicht dient dabei als Eingabe für die nächste Schicht. Auf diese Weise kann das Modell schrittweise komplexere Muster in den Daten lernen.

Frameworks wie Keras übernehmen die Implementierung dieser mathematischen Struktur. Anstatt selbst Gewichtsmatrizen und Aktivierungsfunktionen zu programmieren, beschreibt man nur noch die Architektur des Netzes. Keras erzeugt daraus automatisch die notwendigen TensorFlow-Operationen und kümmert sich später auch um das Training der Gewichte.

### Intuitive Interpretation

Jedes Neuron lernt eine lineare Kombination der Features.

Zum Beispiel könnte ein Neuron lernen:
* grosse Petal Length → eher Virginica
* kleine Petal Length → eher Setosa

Die Gewichte kodieren also letztlich wie wichtig jedes Feature für eine Klasse ist.

In [ ]:
# Erste Architektur: ein Dense-Layer ohne Aktivierung (Ausgabe noch keine Wahrscheinlichkeiten).
inputs = Input(shape=(4,))
fc = Dense(3)(inputs)
model = Model(inputs=inputs, outputs=fc)
model.summary()
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print(model.predict(np.array([[5.1, 3.5, 1.4, 0.2]]), verbose=0))

In [ ]:
# Zweite Architektur: zusätzlicher Softmax-Layer (Klassenausgabe als Wahrscheinlichkeiten).
inputs = Input(shape=(4,))
fc = Dense(3)(inputs)
predictions = Dense(3, activation='softmax')(fc)
model = Model(inputs=inputs, outputs=predictions)
model.summary()
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print(model.predict(np.array([[5.1, 3.5, 1.4, 0.2]]), verbose=0))

# Training

In [ ]:
X = np.array(iris.data)
labels = np.array(iris.target)
print('Daten:', X.shape, labels.shape, '| Label an Index 100:', labels[100])
num_categories = 3
y = to_categorical(labels, num_categories)
print('One-Hot-Label an Index 100:', y[100])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print('Train/Test:', X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
# Originaler Trainingsaufruf mit 500 Epochen und 30 % Validierung.
%time model.fit(X_train, y_train, epochs=500, validation_split=0.3)

# Bewertung

Die Auswertung zeigt die Accuracy und den Loss auf Trainings- und Testdaten. Die Werte werden beim Ausführen berechnet, statt fixe Resultate vorauszusetzen.

Die tatsächlich trainierte Architektur hat **4 Eingaben**, eine Dense-Schicht mit **3 Neuronen** ohne eigene Aktivierung und eine **Softmax-Ausgabeschicht mit 3 Neuronen** (insgesamt **27 Parameter**). Ohne nichtlineare versteckte Aktivierung ist das Modell weiterhin ein **linearer Klassifikator mit Softmax**.

Anschliessend prüfen wir einzelne Vorhersagen anhand der echten Iris-Labels.

In [ ]:
train_loss, train_accuracy = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Training: Loss={train_loss:.4f}, Accuracy={train_accuracy:.2%}')
print(f'Test:     Loss={test_loss:.4f}, Accuracy={test_accuracy:.2%}')
print('Erste Beobachtung:', X[0], '| One-Hot-Label:', y[0])

- - -

## Neuronales Netzwerk verwenden

Beispiel: Ein Datensatz → welche Blume?

Zuerst erzeugt man eine Vorhersage.

In [ ]:
sample = np.array([[5.1, 3.5, 1.4, 0.2]])  # Originalbeispiel / iris.data[0]
prediction = model.predict(sample, verbose=0)
class_id = int(np.argmax(prediction))
print('Wahrscheinlichkeiten:', prediction[0])
print('Klassen:', iris.target_names)
print('Vorhergesagte Blume:', iris.target_names[class_id])

### Gegenprobe und zwei zusätzliche Beispielabfragen

Zuerst vergleichen wir die bekannte Eingabe mit ihrem tatsächlichen Label. Danach fragen wir **zwei weitere Iris-Blumen aus dem Testdatensatz** ab, je eine Versicolor und eine Virginica. Die Gegenprobe vergleicht die vorhergesagte Klasse mit dem tatsächlichen Label; mögliche Fehler werden sichtbar und nicht unterdrückt.

In [ ]:
# Gegenprobe des ursprünglichen Beispiels anhand der Originaldaten.
actual_id = int(iris.target[0])
print(f'Original: erwartet={iris.target_names[actual_id]}, '
      f'erkannt={iris.target_names[class_id]} -> '
      f'{"KORREKT" if class_id == actual_id else "FALSCH"}')

# Zwei zusätzliche Abfragen aus den zurückgehaltenen Testdaten.
for actual_id in (1, 2):  # Versicolor, Virginica
    test_index = np.flatnonzero(np.argmax(y_test, axis=1) == actual_id)[0]
    features = X_test[test_index:test_index + 1]
    probabilities = model.predict(features, verbose=0)[0]
    predicted_id = int(np.argmax(probabilities))
    print(f'{iris.target_names[actual_id]}: Eingabe={features[0]}, '
          f'Wahrscheinlichkeiten={np.round(probabilities, 3)}, '
          f'erwartet={iris.target_names[actual_id]}, '
          f'erkannt={iris.target_names[predicted_id]} -> '
          f'{"KORREKT" if predicted_id == actual_id else "FALSCH"}')